# 2213645 刘嘉硕 回归分析上机作业3

## 1.用向前法建立子集回归模型

In [1]:
library(readxl)
data <- read_excel("C:/Users/lenovo/Screen/TD/5-7_1.xlsx")
head(data)

y,x1,x2,x3,x4,x5
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
271.8,783.35,33.53,40.55,16.66,13.20
264.0,748.45,36.50,36.19,16.46,14.11
238.8,684.45,34.66,37.31,17.66,15.68
230.7,827.80,33.13,32.52,17.50,10.53
251.6,860.45,35.75,33.71,16.40,11.00
257.9,875.15,34.46,34.14,16.28,11.31


In [2]:
# 计算数据的行数
n <- nrow(data)

# 第一次引入
model_0 <- lm(y ~ 1, data = data)
rss_0 <- sum(resid(model_0)^2)

# 为每个模型拟合并计算RSS和F值
models <- list()
rss <- c()
F_values <- c()
x <- c('x1', 'x2', 'x3', 'x4', 'x5')

for (i in 1:5) {
    xi <- x[i]
    formula <- as.formula(paste("y ~ 1 +", xi))
    model <- lm(formula, data = data)
    rss_i <- sum(resid(model)^2)
    F_i <- ((rss_0 - rss_i) / 1) / (rss_i / (n - 1 - 1))
    models[[i]] <- model
    rss <- c(rss, rss_i)
    F_values <- c(F_values, F_i)
}

# 找到F值最大的一项及其对应的F
max_F_index <- which.max(F_values)
max_F <- F_values[max_F_index]
max_variable <- x[max_F_index]
if(max_F > qf(0.05,1,n-1-1,lower.tail = FALSE)){
    cat('第一次引入变量',max_variable,'\n')
}else{
    cat('不能引入剩下的变量。')
}


第一次引入变量 x4 


In [3]:
# 第二次引入
# 为每个模型拟合并计算RSS和F值
models <- list()
rss1 <- c()
F_values <- c()
x <- c('x1', 'x2', 'x3', 'x5')

for (i in 1:4) {
    xi <- x[i]
    formula <- as.formula(paste("y ~ 1 + x4+", xi))
    model <- lm(formula, data = data)
    rss_i <- sum(resid(model)^2)
    F_i <- ((rss[4] - rss_i) / 1) / (rss_i / (n - 2 - 1))
    models[[i]] <- model
    rss1 <- c(rss1, rss_i)
    F_values <- c(F_values, F_i)
}

# 找到F值最大的一项及其对应的F
max_F_index <- which.max(F_values)
max_F <- F_values[max_F_index]
max_variable <- x[max_F_index]
if(max_F > qf(0.05,1,n-2-1,lower.tail = FALSE)){
    cat('第二次引入变量',max_variable,'\n')
}else{
    cat('不能引入剩下的变量。')
}


第二次引入变量 x3 


In [4]:
# 第二次引入
# 为每个模型拟合并计算RSS和F值
models <- list()
rss2 <- c()
F_values <- c()
x <- c('x1', 'x2', 'x5')

for (i in 1:3) {
    xi <- x[i]
    formula <- as.formula(paste("y ~ 1 + x4 + x3 +", xi))
    model <- lm(formula, data = data)
    rss_i <- sum(resid(model)^2)
    F_i <- ((rss1[3] - rss_i) / 1) / (rss_i / (n - 3 - 1))
    models[[i]] <- model
    rss2 <- c(rss2, rss_i)
    F_values <- c(F_values, F_i)
}

# 找到F值最大的一项及其对应的F
max_F_index <- which.max(F_values)
max_F <- F_values[max_F_index]
max_variable <- x[max_F_index]
if(max_F > qf(0.05,1,n-3-1,lower.tail = FALSE)){
    cat('第三次引入变量',max_variable,'\n')
}else{
    cat('第三次不能引入剩下的变量。')
}


第三次不能引入剩下的变量。

前进法结束，选择的变量组合是(x3,x4)

In [5]:
model_best <- lm(y~1+x3+x4,data = data)
a <- coef(model_best)[1]
b <- coef(model_best)[2]
c <- coef(model_best)[3]
regression_equation <- paste("y =", round(a, 4), " +", round(b, 4), "x3",'+',round(c,4),'x4')
cat('得到的回归方程为：',regression_equation)

得到的回归方程为： y = 483.6703  + 4.7963 x3 + -24.215 x4

综上：  
1. 通过前进法，引入变量X3,X4  
2. 得到的回归方程为y = 483.6703  + 4.7963 x3 + -24.215 x4

# 2. 用逐步回归法建立子集回归模型

逐步回归前面部分和前进法一样，通过1中代码，在第三步不能再引入任何变量，逐步回归法的循环也结束了，所以得到的模型和前进法得到的一致

# 3. 应用所有可能子集回归法建立子集回归模型，计算RMS_q值和AIC值，你推荐哪一个子集回归模型，为什么？

In [6]:
variables <- c("x1", "x2", "x3", "x4", "x5")
n <- nrow(data)

# 生成所有可能的子集
all_subsets <- lapply(0:length(variables), function(x) combn(variables, x, simplify = FALSE))

# 初始化存储结果的列表
results <- list()

# 遍历所有子集
subset_index <- 1  # 添加一个子集索引
for (i in seq_along(all_subsets)) {
  for (subset in all_subsets[[i]]) {
    if (length(subset) == 0) {
      formula <- as.formula("y ~ 1")
    } else {
      formula <- as.formula(paste("y ~", paste(subset, collapse = " + ")))
    }
    
    # 拟合模型
    model <- lm(formula, data = data)
    
    # 计算RMS_q
    rss <- sum(resid(model)^2)
    df <- n - length(coef(model))
    rms_q <- sqrt(rss / df)
    
    # 计算AIC
    aic <- AIC(model)
    
    # 存储结果
    results[[as.character(subset_index)]] <- list(
      variables = subset,
      formula = formula,
      rms_q = rms_q,
      aic = aic
    )
    subset_index <- subset_index + 1  # 增加子集索引
  }
}

# 将结果转换为数据框
results_df <- do.call(rbind, lapply(names(results), function(x) {
  data.frame(
    Subset = x,
    Variables = paste(results[[x]]$variables, collapse = ", "),
    Formula = paste(deparse(results[[x]]$formula),collapse = " "),
    RMS_q = results[[x]]$rms_q,
    AIC = results[[x]]$aic
  )
}))
results_df

Subset,Variables,Formula,RMS_q,AIC
<chr>,<chr>,<chr>,<dbl>,<dbl>
1,,y ~ 1,22.898790,266.8837
2,x1,y ~ x1,18.153816,254.3612
3,x2,y ~ x2,23.196521,268.5783
4,x3,y ~ x3,23.172028,268.5170
5,x4,y ~ x4,12.327692,231.9133
6,x5,y ~ x5,21.856064,265.1259
7,"x1, x2",y ~ x1 + x2,17.635664,253.5872
8,"x1, x3",y ~ x1 + x3,17.997947,254.7666
9,"x1, x4",y ~ x1 + x4,12.259123,232.4953


根据 RMSq和AIC值来进行变量选择，综合考虑模型复杂性和RMSq和AIC值尽可能小，选择两个变量（x4, x3）或者三个变量（x2, x3, x4）为最佳选择  
原因：  
(x3,x4)或者(x2,x3,x4)时RMSq和AIC值比较小，再增加变量后RMSq和AIC值变化不大，但是模型的复杂性增加较大